# DuckPD Feature Store: Interactive Walkthrough

Welcome to the **DuckPD Feature Store Walkthrough**! This notebook demonstrates how to connect to large structured Parquet feature stores (such as Hugging Face-hosted datasets like `hf://datasets/hifinab/fdb`), align multi-rate time series with **lookahead protection (point-in-time / ASOF joins)**, access categorical reference tables, and stream training batches with zero Python memory overhead.

### Core Highlights
- **Partition-Mirrored Caching**: Instead of downloading 10s of gigabytes, DuckPD downloads only the partition years you request and projects *only* the feature columns you ask for.
- **Instantaneous Setup**: Creating a store or declaring feature sets executes zero network downloads in the constructor.
- **Transparent JIT Execution**: When you call `.collect()`, `.head()`, or `.to_arrow_batches()`, DuckPD ensures the required partition files are cached locally and runs vectorized queries in DuckDB.
- **Zero Lookahead Bias**: Point-in-time ASOF joins enforce causal observation availability (`availability_delay`).

## Step 1: Import DuckPD and Load Credentials

First, we import DuckPD and read our `HF_TOKEN` from `.env`.

In [1]:
import os
import time
from datetime import timedelta
from pathlib import Path

import duckpd as pd

# Load token from .env if present
token = os.getenv("HF_TOKEN")

cache_dir = Path(".cache/fdb")
print(f"DuckPD version: {pd.__version__}")
print(f"HF_TOKEN configured: {bool(token)}")
print(f"Local cache path: {cache_dir.resolve()}")

DuckPD version: 0.1.3
HF_TOKEN configured: True
Local cache path: /home/hi/duckpd/demo/featurestore_demo/.cache/fdb


## Step 2: Connect to Remote Feature Store & Inspect Catalog

We instantiate `pd.FeatureStore`. Notice that this operation is instantaneous—it loads the compact `catalog.json` schema without touching the multi-gigabyte Parquet data partitions.

In [2]:
store = pd.FeatureStore(
    source="hf://datasets/hifinab/fdb",
    cache=cache_dir,
    token=token,
)

catalog = store.catalog()
print(f"Catalog Name: {catalog.get('name')}")
print(f"Catalog Version: {catalog.get('catalog_version')}")
print(f"Datasets: {[d['name'] for d in catalog.get('datasets', [])]}")
print("\nRegistered Features Sample:")
for feat_name, meta in list(catalog.get("features", {}).items())[:6]:
    delay = meta.get("availability_delay")
    safe = meta.get("lookahead_safe")
    print(f"  - {feat_name:20s} (Delay: {delay}, Safe: {safe})")

Catalog Name: hifinab/fdb
Catalog Version: 1
Datasets: ['ohlcv', 'sma', 'symbology', 'markets']

Registered Features Sample:
  - ohlcv:close          (Delay: PT1M, Safe: True)
  - ohlcv:high           (Delay: PT1M, Safe: True)
  - ohlcv:low            (Delay: PT1M, Safe: True)
  - ohlcv:open           (Delay: PT0S, Safe: True)
  - ohlcv:volume         (Delay: PT1M, Safe: True)
  - sma:sma10            (Delay: PT1M, Safe: True)


/home/hi/duckpd/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3: Access Static Dimension / Reference Tables

Feature stores often contain auxiliary reference data such as ticker symbology or listing exchanges. We access `store.table("symbology")`, which returns a first-class lazy `duckpd.DataFrame`.

In [3]:
symbols = store.table("symbology")
symbols.sort_values("ticker").head(5)

,ticker,isin,cik,company_name,description,market_code
0,000,SE0000000000,0000000001,Example Company 000,Synthetic company record for ticker 000.,XSTO
1,001,SE0000000001,0000000002,Example Company 001,Synthetic company record for ticker 001.,XSTO
2,002,SE0000000002,0000000003,Example Company 002,Synthetic company record for ticker 002.,XSTO
3,003,SE0000000003,0000000004,Example Company 003,Synthetic company record for ticker 003.,XSTO
4,004,SE0000000004,0000000005,Example Company 004,Synthetic company record for ticker 004.,XSTO


## Step 4: Multi-Family Exact Alignment

Exact alignment performs an inner equi-join on matching event timestamps and series keys across datasets. Here, we retrieve minute close prices (`ohlcv:close`) alongside 200-bar moving averages (`sma:sma200`).

In [4]:
t0 = time.perf_counter()
exact_features = store.features(
    features={
        "price": "ohlcv:close",
        "sma200": "sma:sma200",
    },
    start="2024-01-02T08:00:00Z",
    end="2024-01-02T09:00:00Z",
    filters={"ticker": ["001", "002"]},
    alignment="exact",
    order_by=["datetime", "ticker"],
)
exact_df = exact_features.collect()
print(f"Retrieved {len(exact_df)} rows in {time.perf_counter() - t0:.4f}s")
exact_df.head(6)

Retrieved 120 rows in 0.0446s


,datetime,ticker,price,sma200
0,2024-01-02 08:00:00+00:00,001,360.120763,603.463823
1,2024-01-02 08:00:00+00:00,002,184.140033,325.087285
2,2024-01-02 08:01:00+00:00,001,360.022961,602.280158
3,2024-01-02 08:01:00+00:00,002,184.042194,324.367431
4,2024-01-02 08:02:00+00:00,001,359.284077,601.077755
5,2024-01-02 08:02:00+00:00,002,183.859903,323.647392


## Step 5: Point-in-Time (ASOF) Alignment with Availability Delays

In financial and event-driven machine learning, training models on data before it was physically available introduces **lookahead bias**.

In this catalog:
- `ohlcv:open` has availability delay `PT0S` (available immediately at bar open).
- `ohlcv:close` has availability delay `PT1M` (the close of the 08:00 bar is only known once the bar ends at 08:01).

With `alignment="point_in_time"` and `spine="ohlcv"`, DuckPD compiles a vectorized `ASOF LEFT JOIN`.

In [5]:
pit_features = store.features(
    features={
        "open": "ohlcv:open",
        "close": "ohlcv:close",
        "sma50": "sma:sma50",
        "sma200": "sma:sma200",
    },
    start="2024-01-02T08:00:00Z",
    end="2024-01-02T08:06:00Z",
    filters={"ticker": ["001"]},
    alignment="point_in_time",
    spine="ohlcv",
    order_by=["datetime"],
)
pit_df = pit_features.collect()
print("Notice: At 08:00, 'close' is NaN because the 08:00 close value is only known at 08:01!")
pit_df

Notice: At 08:00, 'close' is NaN because the 08:00 close value is only known at 08:01!


,datetime,ticker,open,close,sma50,sma200
0,2024-01-02 08:00:00+00:00,001,359.552157,610.433563,606.300274,604.652947
1,2024-01-02 08:01:00+00:00,001,360.069019,360.120763,601.362335,603.463823
2,2024-01-02 08:02:00+00:00,001,359.913349,360.022961,596.418132,602.280158
3,2024-01-02 08:03:00+00:00,001,359.330338,359.284077,591.458914,601.077755
4,2024-01-02 08:04:00+00:00,001,358.865571,358.858458,586.502272,599.870908
5,2024-01-02 08:05:00+00:00,001,359.002459,359.086650,581.561604,598.666293


## Step 6: Seamless DuckPD DataFrame Composition

Because `store.features()` returns a native `duckpd.DataFrame`, you can chain pandas-style operations lazily:
- Compute new indicators with `.assign()`
- Filter on signal conditions
- Merge with categorical symbology tables

In [6]:
signals = pit_features.assign(
    trend=lambda df: df["close"] / df["sma200"],
    spread=lambda df: df["close"] - df["open"],
).merge(symbols, on="ticker", how="left")

signals_df = signals.collect()
signals_df[["datetime", "ticker", "company_name", "close", "trend", "spread"]]

,datetime,ticker,company_name,close,trend,spread
0,2024-01-02 08:00:00+00:00,001,Example Company 001,610.433563,1.009560,250.881406
1,2024-01-02 08:01:00+00:00,001,Example Company 001,360.120763,0.596756,0.051744
2,2024-01-02 08:02:00+00:00,001,Example Company 001,360.022961,0.597767,0.109612
3,2024-01-02 08:03:00+00:00,001,Example Company 001,359.284077,0.597733,-0.046260
4,2024-01-02 08:04:00+00:00,001,Example Company 001,358.858458,0.598226,-0.007113
5,2024-01-02 08:05:00+00:00,001,Example Company 001,359.086650,0.599811,0.084190


## Step 7: Zero-Copy Arrow Batch Streaming for Machine Learning

For massive training sets that span multiple months or years, load chunked batches over consecutive time windows using `.feature_batches()` and stream directly to PyArrow record batches.

In [7]:
batch_count = 0
row_count = 0
t0 = time.perf_counter()

for window_df in store.feature_batches(
    exact_features,
    window=timedelta(minutes=30),
    start="2024-01-02T08:00:00Z",
    end="2024-01-02T09:00:00Z",
):
    with window_df.to_arrow_batches(batch_size=10_000) as reader:
        for arrow_batch in reader:
            batch_count += 1
            row_count += arrow_batch.num_rows

elapsed = time.perf_counter() - t0
print(f"Streamed {batch_count} window batches ({row_count} rows) in {elapsed:.4f}s")

Streamed 2 window batches (120 rows) in 0.0520s


## Step 8: Inspect Local Partition-Mirrored Cache Footprint

Observe the local cache directory: the downloaded files mirror the remote directory layout exactly (`<dataset>/year=<year>/data.parquet`) and contain only the requested columns, keeping disk usage minimal.

In [8]:
print(f"Cache Directory: {cache_dir.resolve()}")
for p in sorted(cache_dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(cache_dir)}: {p.stat().st_size / 1e6:.2f} MB")

Cache Directory: /home/hi/duckpd/demo/featurestore_demo/.cache/fdb
  catalog.json: 0.00 MB
  ohlcv/year=2023/data.parquet: 226.61 MB
  ohlcv/year=2024/data.parquet: 226.59 MB
  sma/year=2023/data.parquet: 220.01 MB
  sma/year=2024/data.parquet: 220.06 MB
  symbols/data.parquet: 0.00 MB
